In [10]:
import pandas as pd
import numpy as np


df = pd.read_csv("./data/log_sensor_fix.csv")
print(f"Data: {df.shape[0]} baris, {df.shape[1]} kolom")
df.head()

Data: 257 baris, 9 kolom


,Timestamp,soil_moisture,soil_temperature,air_temperature,air_humidity,nitrogen,fosfor,kalium,ec
0,2026-07-05T09:07:11Z,23.7,8.4,24.4,75.6,128,232,163,3829
1,2026-07-05T09:08:12Z,23.5,8.4,24.4,75.5,128,232,163,3831
2,2026-07-05T09:09:12Z,23.3,8.4,24.4,75.9,127,232,163,3833
3,2026-07-05T09:10:15Z,23.2,8.4,24.5,76.2,126,232,162,3835
4,2026-07-05T09:11:15Z,23.0,8.4,24.5,76.0,125,232,162,3836


# Cek data

In [11]:
df[["soil_moisture", "soil_temperature", "air_temperature",
    "air_humidity", "nitrogen", "fosfor", "kalium", "ec"]].describe().round(2)

,soil_moisture,soil_temperature,air_temperature,air_humidity,nitrogen,fosfor,kalium,ec
count,257.00,257.00,257.00,257.00,257.00,257.00,257.00,257.0
mean,15.59,7.93,22.29,77.75,97.80,204.44,153.06,3914.3
std,7.10,0.58,5.83,17.30,22.63,45.77,11.49,74.8
min,9.90,7.60,15.20,40.00,83.00,147.00,146.00,3293.0
25%,11.00,7.60,18.00,71.90,85.00,169.00,147.00,3889.0
50%,13.50,7.80,20.60,87.30,91.00,194.00,150.00,3937.0
75%,18.00,8.10,24.50,89.90,101.00,229.00,156.00,3963.0
max,74.60,14.00,34.30,93.20,283.00,316.00,274.00,3975.0


# Normalisasi Skala Sensor

In [12]:
# ec: sensor output µS/cm, konversi ke mS/cm (bagi 100)

df["ec"] = df["ec"] / 100.0       

# df["kalium"] = df["kalium"] / 3.5

print("Setelah normalisasi:")
print(df[["nitrogen", "fosfor", "kalium", "ec"]].describe().round(2))

Setelah normalisasi:
       nitrogen  fosfor  kalium      ec
count    257.00  257.00  257.00  257.00
mean      97.80  204.44  153.06   39.14
std       22.63   45.77   11.49    0.75
min       83.00  147.00  146.00   32.93
25%       85.00  169.00  147.00   38.89
50%       91.00  194.00  150.00   39.37
75%      101.00  229.00  156.00   39.63
max      283.00  316.00  274.00   39.75


# Tambah Kolom Fase (butuh plant_age)

In [13]:
TANGGAL_TANAM = pd.Timestamp("2026-06-15", tz="UTC")

df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df["plant_age"] = (df["Timestamp"] - TANGGAL_TANAM).dt.days

def get_fase(age):
    if age <= 30:   return 0    # establishment
    elif age <= 55: return 1    # vegetatif
    elif age <= 75: return 2    # berbunga (initial flowering - fruit set)
    else:           return 3    # pematangan (fruit development & maturation)

df["fase"] = df["plant_age"].apply(get_fase)

FASE_NAMES = {0: "establishment", 1: "vegetatif", 2: "berbunga", 3: "pematangan"}

print(df[["Timestamp", "plant_age", "fase"]].head())
print("\nDistribusi fase:")
for code, name in FASE_NAMES.items():
    n = (df["fase"] == code).sum()
    print(f"  {code} = {name:15s} {n}")

                  Timestamp  plant_age  fase
0 2026-07-05 09:07:11+00:00         20     0
1 2026-07-05 09:08:12+00:00         20     0
2 2026-07-05 09:09:12+00:00         20     0
3 2026-07-05 09:10:15+00:00         20     0
4 2026-07-05 09:11:15+00:00         20     0

Distribusi fase:
  0 = establishment   257
  1 = vegetatif       0
  2 = berbunga        0
  3 = pematangan      0


# Rasio Ideal NPK per Fase (dari Haifa)

In [14]:
# Rasio N:P2O5:K2O ideal per fase (Haifa Crop Guide)
RASIO_IDEAL = {
    0: (1, 2, 1),   # establishment - P dominan (root development)
    1: (1, 1, 1),   # vegetatif - seimbang
    2: (2, 1, 3),   # berbunga - K dominan, P turun
    3: (2, 1, 3),   # pematangan - K dominan
}

# Batas absolut per unsur (mg/kg) - sesuaikan skala sensormu
BATAS_MIN = {"n": 40, "p": 50, "k": 60}
BATAS_MAX = {"n": 150, "p": 150, "k": 200}

print("Rasio ideal & batas siap.")

Rasio ideal & batas siap.


# Labelling dataset pupuk

In [ ]:
LABEL_NAMES = {
    0: "Tidak perlu",
    1: "Urea/ZA",              # N kurang
    2: "SP-36",                # P kurang
    3: "KCl",                  # K kurang
    4: "Urea/ZA + SP-36",      # N,P kurang
    5: "Urea/ZA + KCl",        # N,K kurang
    6: "SP-36 + KCl",          # P,K kurang
    7: "Urea/ZA + SP-36 + KCl",# semua kurang
    8: "NPK 15-15-15",         # maintenance
    9: "Kurangi pemupukan N",  # N berlebih
    10: "Flush air (EC/nutrisi tinggi)",  # over/salinitas
}

def label_pupuk(row):
    n, p, k = row["nitrogen"], row["fosfor"], row["kalium"]
    ec = row["ec"]
    fase = row["fase"]   # sekarang int (0-3)

    # --- Cek kelebihan dulu ---
    if ec > 4.0:
        return 10
    if n > BATAS_MAX["n"] and k > BATAS_MAX["k"]:
        return 10
    if n > BATAS_MAX["n"]:
        return 9

    # --- Defisiensi berdasarkan rasio fase ---
    rn, rp, rk = RASIO_IDEAL[fase]  
    total_ratio = rn + rp + rk

    total_npk = n + p + k
    if total_npk == 0:
        return 7

    prop_n, prop_p, prop_k = n/total_npk, p/total_npk, k/total_npk
    ideal_n, ideal_p, ideal_k = rn/total_ratio, rp/total_ratio, rk/total_ratio

    TOLERANSI = 0.7
    n_low = (prop_n < ideal_n * TOLERANSI) or (n < BATAS_MIN["n"])
    p_low = (prop_p < ideal_p * TOLERANSI) or (p < BATAS_MIN["p"])
    k_low = (prop_k < ideal_k * TOLERANSI) or (k < BATAS_MIN["k"])

    if   n_low and p_low and k_low: return 7
    elif n_low and p_low:           return 4
    elif n_low and k_low:           return 5
    elif p_low and k_low:           return 6
    elif n_low:                     return 1
    elif p_low:                     return 2
    elif k_low:                     return 3
    else:                           return 0

print("Fungsi label_pupuk() siap.")

Fungsi label_pupuk() siap.


In [16]:
df["recommendation"] = df.apply(label_pupuk, axis=1)
df["recommendation_label"] = df["recommendation"].map(LABEL_NAMES)

print("Distribusi rekomendasi:")
dist = df["recommendation"].value_counts().sort_index()
for code, count in dist.items():
    print(f"  {code} = {LABEL_NAMES[code]:35s} {count:5d} ({count/len(df)*100:.1f}%)")

Distribusi rekomendasi:
  10 = Flush air (EC/nutrisi tinggi)         257 (100.0%)


# Preview Hasil

In [17]:
df["fase_label"] = df["fase"].map(FASE_NAMES)
df[["Timestamp", "nitrogen", "fosfor", "kalium", "ec",
    "fase_label", "recommendation_label"]].head(20)

,Timestamp,nitrogen,fosfor,kalium,ec,fase_label,recommendation_label
0,2026-07-05 09:07:11+00:00,128,232,163,38.29,establishment,Flush air (EC/nutrisi tinggi)
1,2026-07-05 09:08:12+00:00,128,232,163,38.31,establishment,Flush air (EC/nutrisi tinggi)
2,2026-07-05 09:09:12+00:00,127,232,163,38.33,establishment,Flush air (EC/nutrisi tinggi)
3,2026-07-05 09:10:15+00:00,126,232,162,38.35,establishment,Flush air (EC/nutrisi tinggi)
4,2026-07-05 09:11:15+00:00,125,232,162,38.36,establishment,Flush air (EC/nutrisi tinggi)
5,2026-07-05 09:17:20+00:00,118,232,161,38.45,establishment,Flush air (EC/nutrisi tinggi)
6,2026-07-05 09:22:24+00:00,117,231,160,38.51,establishment,Flush air (EC/nutrisi tinggi)
7,2026-07-05 09:27:24+00:00,115,230,160,38.56,establishment,Flush air (EC/nutrisi tinggi)
8,2026-07-05 09:32:24+00:00,113,230,159,38.60,establishment,Flush air (EC/nutrisi tinggi)
9,2026-07-05 09:37:24+00:00,112,229,159,38.64,establishment,Flush air (EC/nutrisi tinggi)


# Simpan dataset

In [18]:
sensor_cols = ["soil_moisture", "soil_temperature", "air_temperature",
               "air_humidity", "nitrogen", "fosfor", "kalium", "ec"]

# Dataset pupuk: 7 fitur + label
cols_pupuk = ["nitrogen", "fosfor", "kalium", "plant_age", "fase", "ec", "soil_moisture", "recommendation"]
df[cols_pupuk].to_csv("data/dataset_pupuk.csv", index=False)

print("Tersimpan: dataset_pupuk.csv")
print(f"Total: {len(df)} baris")

Tersimpan: dataset_pupuk.csv
Total: 257 baris
